In [1]:
import re
from collections import defaultdict
from typing import Tuple, Dict

import numpy as np
from scipy.io import loadmat
from scipy.sparse import csc_matrix
import cobra

In [2]:
# --- precompile regexes once ---
_RE_MET = re.compile(r"^(.*)\[([A-Za-z0-9_]+)\]$")
_RE_GENE_SUFFIX = re.compile(r"\.(\d+)$")
_RE_GPR_SUFFIX = re.compile(r"(\d+)\.\d+")


def normalize_gene_id(gid: str) -> str:
    """Strip trailing .digits from gene IDs (e.g. '4830.1' -> '4830')."""
    if gid is None:
        return ""
    gid = str(gid).strip()
    return _RE_GENE_SUFFIX.sub("", gid)


def normalize_gpr(gpr_str) -> str:
    """Make sure GPR is a cleaned string and remove numeric suffixes."""
    if gpr_str is None:
        return ""
    s = str(gpr_str).strip()
    if not s or s in {"[]", "nan", "None"}:
        return ""
    # convert "123.1" -> "123"
    s = _RE_GPR_SUFFIX.sub(r"\1", s)
    # remove braces
    return s.replace("{", "").replace("}", "")


def transform_matlab_met_id(raw_met: str) -> Tuple[str, str]:
    """Transform 'met[c]' -> ('met_c', 'c'). Default compartment 'c'."""
    if raw_met is None:
        return "unknown", "c"
    raw = str(raw_met)
    m = _RE_MET.match(raw)
    if m:
        base, comp = m.group(1), m.group(2)
        return f"{base}_{comp}", comp
    return raw, "c"


def cobramat2cobrapy(cobramat, name: str = None) -> cobra.Model:
    """Optimized conversion from MATLAB COBRA struct to COBRApy Model.

    - Uses S.tocoo() to iterate nonzero entries only.
    - Avoids nested full-range loops over matrix rows.
    - Ensures unique metabolite IDs by stamping duplicates.
    """
    model_id = name or str(getattr(cobramat, "description", "unnamed_model"))
    model = cobra.Model(model_id)

    # --- core arrays ---
    rxns = np.array(cobramat.rxns, dtype=str)
    mets = np.array(cobramat.mets, dtype=str)
    S = csc_matrix(cobramat.S)
    lb = np.array(cobramat.lb, dtype=float)
    ub = np.array(cobramat.ub, dtype=float)
    cvec = np.array(cobramat.c, dtype=float)
    b = np.array(getattr(cobramat, "b", np.zeros(len(mets))), dtype=float)

    # optional metadata
    metFormulas = getattr(cobramat, "metFormulas", None)
    metNames = getattr(cobramat, "metNames", None)
    grRules = getattr(cobramat, "grRules", None)
    genes = getattr(cobramat, "genes", None)
    rxnNames = getattr(cobramat, "rxnNames", None)
    subSystems = getattr(cobramat, "subSystems", None)

    # --- build metabolite objects efficiently ---
    met_objs: Dict[str, cobra.Metabolite] = {}
    seen_ids = defaultdict(int)
    existing_met_ids = set()

    for i, raw_met in enumerate(mets):
        new_id, comp = transform_matlab_met_id(raw_met)
        # ensure unique ids deterministically
        if new_id in existing_met_ids:
            seen_ids[new_id] += 1
            unique_id = f"{new_id}__dup{seen_ids[new_id]}"
        else:
            unique_id = new_id
            seen_ids[new_id] = 0
        existing_met_ids.add(unique_id)

        met = cobra.Metabolite(unique_id, compartment=comp)

        # attach formula & name if available (guarded)
        if metFormulas is not None and i < len(metFormulas):
            formula = str(metFormulas[i]).strip()
            if formula and formula not in {"[]", "nan", "None"}:
                met.formula = formula
        if metNames is not None and i < len(metNames):
            name_str = str(metNames[i]).strip()
            if name_str and name_str not in {"[]", "nan", "None"}:
                met.name = name_str

        # key by the original raw MATLAB string (ensures S mapping works)
        met_objs[str(raw_met)] = met

    # add all metabolites once
    model.add_metabolites(list(met_objs.values()))

    # --- prepare stoichiometry grouped by reaction using sparse COO view ---
    coo = S.tocoo()
    # group nonzeros by column (reaction index)
    col_to_entries = defaultdict(list)
    for row_idx, col_idx, value in zip(coo.row, coo.col, coo.data):
        col_to_entries[col_idx].append((row_idx, value))

    # --- create reactions and attach stoichiometry using grouped entries ---
    for j, rxn_id in enumerate(rxns):
        rxn = cobra.Reaction(rxn_id)
        if rxnNames is not None and j < len(rxnNames):
            rxn.name = str(rxnNames[j]).strip()
        if subSystems is not None and j < len(subSystems):
            rxn.subsystem = str(subSystems[j]).strip()

        rxn.lower_bound = float(lb[j])
        rxn.upper_bound = float(ub[j])

        # add reaction to model before assigning stoich and objective
        model.add_reactions([rxn])

        stoich = {}
        entries = col_to_entries.get(j)
        if entries:
            for row_idx, coeff in entries:
                raw_met = str(mets[row_idx])
                met_obj = met_objs.get(raw_met)
                if met_obj is None:
                    # fallback: transform and create if missing
                    fallback_id, fallback_comp = transform_matlab_met_id(raw_met)
                    if fallback_id not in existing_met_ids:
                        mtemp = cobra.Metabolite(fallback_id, compartment=fallback_comp)
                        model.add_metabolites([mtemp])
                        met_objs[raw_met] = mtemp
                        existing_met_ids.add(fallback_id)
                        met_obj = mtemp
                    else:
                        met_obj = model.metabolites.get_by_id(fallback_id)
                stoich[met_obj] = float(coeff)

        if stoich:
            rxn.add_metabolites(stoich)
        else:
            # mark empty stoichiometry to avoid boundary-type heuristic failures
            rxn.notes["empty_stoichiometry"] = True
            try:
                rxn._compartments = {"e"}  # best-effort extracellular tag
            except Exception:
                pass

        # set objective coefficient if present
        if j < cvec.size and float(cvec[j]) != 0.0:
            try:
                rxn.objective_coefficient = float(cvec[j])
            except Exception:
                pass

        # set GPR safely
        if grRules is not None and j < len(grRules):
            rule = normalize_gpr(grRules[j])
            if rule:
                try:
                    rxn.gene_reaction_rule = rule
                except Exception:
                    # log once per problematic rxn
                    print(f"⚠ Could not set GPR for {rxn_id}: {rule}")

    # --- genes: add to model (fast set membership) ---
    if genes is not None:
        existing_gene_ids = {g.id for g in model.genes}
        for g in np.array(genes, dtype=str):
            ng = normalize_gene_id(g)
            if ng and ng not in existing_gene_ids:
                model.genes.append(cobra.Gene(ng))
                existing_gene_ids.add(ng)

    # --- set model objective from cvec (if needed) ---
    try:
        if np.any(cvec):
            pos_idx = np.where(np.array(cvec, dtype=float) > 0)[0]
            if pos_idx.size > 0:
                obj_map = {}
                for idx in pos_idx:
                    rxn_obj = model.reactions.get_by_id(rxns[idx])
                    obj_map[rxn_obj] = 1.0
                model.objective = obj_map
    except Exception:
        print("⚠ Warning: could not set objective automatically")

    model.id = model_id
    model.description = model_id
    return model


In [3]:
matfile = "models/Zielinski_et_al_2017/code/modelsAll.mat"
print(f"Loading matlab models from {matfile}")
data = loadmat(matfile, struct_as_record=False, squeeze_me=True)
models = data["modelsAll"]

print("Loading matlab structure with all NCI cell line models")
cell_line_labels = []
with open("models/Zielinski_et_al_2017/cell_line_labels.txt") as fh:
    cell_line_labels = [l.rstrip() for l in fh]

assert len(models) == len(cell_line_labels)

Loading matlab models from models/Zielinski_et_al_2017/code/modelsAll.mat


In [4]:
cobra_model_list = []
c = 1
print(f"Converting {len(models)} models")
for i,m in enumerate(models):
    cobra_model = cobramat2cobrapy(m)
    cobra_model.id = cell_line_labels[i]
    cobra_model_list.append(cobra_model)
    print(f"Parssing {cobra_model.id} ({c} from {len(models)})")
    c += 1

Converting 59 models
Parssing LINE_UACC_257 (1 from 59)
Parssing LINE_OVCAR_8 (2 from 59)
Parssing LINE_OVCAR_5 (3 from 59)
Parssing LINE_SF_295 (4 from 59)
Parssing LINE_A549_ATCC (5 from 59)
Parssing LINE_RXF_393 (6 from 59)
Parssing LINE_CAKI_1 (7 from 59)
Parssing LINE_MDA_MB_435 (8 from 59)
Parssing LINE_EKVX (9 from 59)
Parssing LINE_NCI_H23 (10 from 59)
Parssing LINE_HCC_2998 (11 from 59)
Parssing LINE_NCI_ADR_RES (12 from 59)
Parssing LINE_HT29 (13 from 59)
Parssing LINE_M14 (14 from 59)
Parssing LINE_HCT_116 (15 from 59)
Parssing LINE_HOP_62 (16 from 59)
Parssing LINE_786_O (17 from 59)
Parssing LINE_NCI_H322M (18 from 59)
Parssing LINE_BT_549 (19 from 59)
Parssing LINE_MALME_3M (20 from 59)
Parssing LINE_TK_10 (21 from 59)
Parssing LINE_UO_31 (22 from 59)
Parssing LINE_T_47D (23 from 59)
Parssing LINE_MCF7 (24 from 59)
Parssing LINE_NCI_H460 (25 from 59)
Parssing LINE_SK_OV_3 (26 from 59)
Parssing LINE_SK_MEL_5 (27 from 59)
Parssing LINE_SNB_19 (28 from 59)
Parssing LINE_SF_5

In [5]:
m0 = cobra_model_list[0]

print(f"Inspecting exchange fluxes bounds for cell line model {m0.id}")
print(m0.id)
for r in m0.exchanges:
    print(r.id, r.bounds)

Inspecting exchange fluxes bounds for cell line model LINE_UACC_257
LINE_UACC_257
EX_ala_L(e) (0.008191710808237964, 0.011074541736715236)
EX_arg_L(e) (-0.013883460976794316, -0.004055325638433218)
EX_asn_L(e) (-0.008311610605135638, -0.006183048814401046)
EX_asp_L(e) (-0.004382247395518676, -0.0038697656395328952)
EX_chol(e) (-1000.0, 1000.0)
EX_cit(e) (0.0030285232167537113, 0.003227168001705683)
EX_cl(e) (-1000.0, 1000.0)
EX_co2(e) (0.0, 1000.0)
EX_glc(e) (-0.43051118477269784, -0.42885317023069547)
EX_gln_L(e) (-0.11174203599983473, -0.1063891582481125)
EX_glu_L(e) (0.00802518036235933, 0.009337373822980518)
EX_gly(e) (-0.0018480416568345173, 0.00036623154549854646)
EX_h(e) (-1000.0, 1000.0)
EX_h2o(e) (-1000.0, 1000.0)
EX_ile_L(e) (-0.016957233128930178, -0.016020903201079177)
EX_k(e) (-1000.0, 1000.0)
EX_lac_L(e) (0.4738567083048213, 0.4904368549353552)
EX_leu_L(e) (-0.018863412677622288, -0.018842833997889295)
EX_lys_L(e) (-0.012816841977797947, -0.012590839653906822)
EX_na1(e) (

In [6]:
print(f"Optimizing default objective function for cell line model {m0.id}")
m0.optimize()
summary_target_nadph = m0.summary()
summary_target_nadph

Optimizing default objective function for cell line model LINE_UACC_257


Metabolite,Reaction,Flux,C-Number,C-Flux
Tyr_ggn_c,DM_Tyr_ggn_c_,0.000584,0,0.00%
gudac_c,DM_gudac_c_,0.00791,3,0.62%
arg_L_e,EX_arg_L(e),0.01388,6,2.17%
asn_L_e,EX_asn_L(e),0.008312,4,0.87%
asp_L_e,EX_asp_L(e),0.004382,4,0.46%
chol_e,EX_chol(e),0.0006024,5,0.08%
glc_D_e,EX_glc(e),0.4305,6,67.41%
gln_L_e,EX_gln_L(e),0.1117,5,14.58%
gly_e,EX_gly(e),0.001848,2,0.10%
h2o_e,EX_h2o(e),1.26,0,0.00%


In [7]:
print(f"Optimizing biomass_NCI60 objective function for cell line model {m0.id}")
for r in m0.reactions:
    if r.objective_coefficient == 0:
        continue
    r.objective_coefficient = 0

m0.reactions.biomass_NCI60.objective_coefficient = 1
m0.optimize()
summary_target_biomass = m0.summary()
summary_target_biomass

Optimizing biomass_NCI60 objective function for cell line model LINE_UACC_257


Metabolite,Reaction,Flux,C-Number,C-Flux
Tyr_ggn_c,DM_Tyr_ggn_c_,0.0007255,0,0.00%
gudac_c,DM_gudac_c_,0.007892,3,0.64%
arg_L_e,EX_arg_L(e),0.004055,6,0.66%
asn_L_e,EX_asn_L(e),0.006183,4,0.67%
asp_L_e,EX_asp_L(e),0.00387,4,0.42%
chol_e,EX_chol(e),0.0007483,5,0.10%
glc_D_e,EX_glc(e),0.4289,6,69.46%
gln_L_e,EX_gln_L(e),0.1064,5,14.36%
ile_L_e,EX_ile_L(e),0.01602,6,2.59%
leu_L_e,EX_leu_L(e),0.01884,6,3.05%
